# Phase-gradient quantum Fourier transform

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2023 Google LLC, Apache-2.0. [Locked source notebook](https://github.com/quantumlib/Qualtran/blob/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/bloqs/qft/qft_phase_gradient.ipynb).

In [ ]:
from IPython.display import SVG, display
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from examples.algorithms.field_addition_catalog import field_signature_svg

## Fourier phases from reversible addition

Prepare an $n$-qubit gradient with amplitudes proportional to $e^{-2\pi i k/2^n}$. Adding into that register kicks phase back onto the data and preserves the eigenstate. The transform recurses on the left half, accumulates the bit-reversed left half times the right half into the full gradient, then recurses on the right. Only the outer circuit optionally reverses the output. A shorter gradient changes the accuracy and belongs to the approximate QFT. [Derivation](https://algassert.com/post/1620).

In [ ]:
from flagquantum.algorithms.primitives.qft_phase_gradient_resource_model import (
    source_phase_gradient_qft_signature, source_phase_gradient_qft_graph, source_phase_gradient_qft_formula,
)
from examples.algorithms.phase_gradient_qft_catalog import phase_gradient_qft_label

### Example instance

In [ ]:
width = 3
source_phase_gradient_qft_signature(width)

#### Graphical signature

Both ports are through registers. The data is unsigned and the gradient is an unsigned fractional word.

In [ ]:
display(SVG(field_signature_svg(((
    "Phase-gradient QFT", source_phase_gradient_qft_signature(width), {},
),))))

### Call graph and modeled resources

The declared graph counts identical recursive child keys once and defaults each child to reversal. The executable circuit visits both children with reversal disabled. The symbolic graph supplies the source logarithmic bound.

In [ ]:
from sympy import Symbol
graph = source_phase_gradient_qft_graph(width, max_depth=1)
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls, title=phase_gradient_qft_label))))
print("Source model:", source_phase_gradient_qft_formula(width))
print("Source symbolic bound:", source_phase_gradient_qft_formula(Symbol("n")))

### Execute through append

Append the prepared-gradient transform to a dynamic circuit. The product accumulation implementation uses clean work and measured cleanup. The emitted counts below include that implementation; the source model above declares costs for the arithmetic leaf.

In [ ]:
from collections import Counter
import torch
from flagquantum.algorithms.primitives.qft import qft
from flagquantum.algorithms.primitives.qft_phase_gradient import qft_phase_gradient_ir, qft_phase_gradient_requirements
from flagquantum.circuit import Circuit
from flagquantum.compiler.registers.phase.phase_gradient import phase_gradient_state_ir
from flagquantum.dynamic import DynamicCircuit
from flagquantum.runtime.dynamic.execution import run_dynamic
from examples.algorithms.binary_field_arithmetic import FieldOperation

extra, slots = qft_phase_gradient_requirements(width)
data, gradient = tuple(range(width)), tuple(range(width, 2 * width))
work = tuple(range(2 * width, 2 * width + extra))
body = qft_phase_gradient_ir(data, gradient, work, tuple(range(slots)))
print("Emitted transform:", Counter(op.name for op in body))
print("Clean work wires:", extra)
gates = (*phase_gradient_state_ir(gradient, exponent=-1), *body,
         *phase_gradient_state_ir(gradient, exponent=-1, inverse=True))
n = 2 * width + extra
circuit = DynamicCircuit(n, dtype=torch.complex128)
circuit.append(FieldOperation(n, gates))
result = run_dynamic(circuit, shots=4, seed=21, strategy="batched")
reference = Circuit(width, dtype=torch.complex128)
reference.append(FieldOperation(width, tuple(qft(width).to_ir().instructions)))
expected = torch.zeros(2**n, dtype=torch.complex128)
expected[::2**(width + extra)] = reference.state().flatten()
torch.testing.assert_close(result.final_states.reshape(4, -1), expected.expand(4, -1))